In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv('SolCbio3_descs_fps.csv') #import filtered descriptors
data.drop(columns=['smiles', 'mol'], inplace=True) #remove unwanted columns

# Create a new column 'output' by mapping 'dev' column
data['output'] = data['dev'].map({'NO': 0, 'YES': 1})

# Delete the original 'dev' column
data.drop(columns=['dev'], inplace=True)


X = data.drop('output', axis=1)
y = data['output']

from sklearn.model_selection import StratifiedShuffleSplit


sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in sss.split(X, y):
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Now you have stratified samples in X_train, X_test, y_train, and y_test

print(f'Training set (80%): {X_train.shape}')
print(f'Test set (20%): {X_test.shape}')
print(f'Accurate predictions of GSE in test set: {100*round(y_test.value_counts()[0]/len(y_test),3)} %')
print(f'Accurate predictions of GSE in training set: {100*round(y_train.value_counts()[0]/len(y_train),3)} %')

Training set (80%): (2192, 1125)
Test set (20%): (548, 1125)
Accurate predictions of GSE in test set: 65.3 %
Accurate predictions of GSE in training set: 65.4 %


In [2]:
from sklearn.ensemble import RandomForestClassifier


# Initialize a logistic regression model with increased max_iter
model = RandomForestClassifier(random_state=42)

from sklearn.feature_selection import RFECV
from sklearn.svm import LinearSVC

# Your RFECV setup
rfecv = RFECV(estimator=model, step=50, cv=3, scoring='accuracy', n_jobs=-1)
rfecv.fit(X_train, y_train)

# Get selected feature names
feature_names = [data.columns[i] for i in range(X_train.shape[1])]
selected_feature_names = np.array(feature_names)[rfecv.support_]

# Filter your original DataFrame to keep only selected features
df_selected = data[selected_feature_names]

print(f"Original df shape: {data.shape}")
print(f"Filtered df shape: {df_selected.shape}")
print(f"Number of features selected: {len(selected_feature_names)}")





Original df shape: (2740, 1126)
Filtered df shape: (2740, 75)
Number of features selected: 75


In [3]:
data = pd.read_csv('SolCbio3_descs_fps.csv') #import filtered descriptors
df_selected['dev'] = data['dev']
df_selected['smiles'] = data['smiles']
df_selected.to_csv('RFE_SolCbio3_descs_fps.csv', index=False)